# RFT Code Repair: MAI-Code-1.1-Flash (preview)

Use the **Microsoft Foundry Project SDK** to train `MAI-Code-1.1-Flash` with reinforcement fine-tuning (RFT). The model proposes Python code repairs, and a model-based grader rewards functional, minimal changes.

The workflow is: inspect the data, load the grader, connect to your project, upload two files, submit an RFT job, and inspect its training metrics. MAI-Code-1.1-Flash supports **managed RFT only**, not SFT, DPO, or interactive training.

Follow the [setup guide](../../../GETTING_STARTED.md) and select the **Foundry managed** kernel. Run from this demo directory or its `notebooks/` directory. Sections 1-3 are local; Sections 4 onward use Foundry. Do not use **Run All** as an offline check.

> **Training costs:** This workflow uses paid cloud resources and may incur charges.

## 1. Set the model and paths

Use the versioned training model ID below. Confirm preview access, model and grader availability, region support, and quota in your project before running the remote sections.

In [ ]:
import json
import os
import time
from pathlib import Path

MODEL = "MAI-Code-1.1-Flash-2026-09-15"

DEMO_ROOT = Path.cwd().resolve()
if DEMO_ROOT.name == "notebooks":
    DEMO_ROOT = DEMO_ROOT.parent

TRAIN_PATH = DEMO_ROOT / "data" / "preserved" / "code-repair-rft-training.jsonl"
VALIDATION_PATH = DEMO_ROOT / "data" / "preserved" / "code-repair-rft-validation.jsonl"
OUTPUT_DIR = DEMO_ROOT / "outputs"

## 2. Inspect a code-repair task

The [dataset](../data/README.md) has **50 training** and **15 validation** tasks. Each record contains:

| Field | Purpose |
|---|---|
| `messages` | Developer instructions and the user's code-repair request, sent to the model. |
| `reference_patch` | A reference diff for the grader, not the model. |
| `test_expectations` | Expected behavior for the grader to consider. |
| `requirements` | Repair constraints and edge cases for the grader. |

There is no assistant answer in the data: RFT generates candidate repairs and scores them. The prompt asks for JSON containing `patch`, `explanation`, and `tests`. Let's inspect the first example.

In [ ]:
training_rows = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines()]
validation_rows = [json.loads(line) for line in VALIDATION_PATH.read_text(encoding="utf-8").splitlines()]

print(f"Training examples: {len(training_rows)}; validation examples: {len(validation_rows)}")
print(json.dumps(training_rows[0], indent=2))

## 3. Load the score-model grader

The [grader definition](../graders/code_repair_quality.json) uses `o3-mini-2025-01-31` to return a score from **0 to 1**, with a pass threshold of **0.7**. Its rubric weights correctness (55%), requirements and edge cases (20%), minimality and regression safety (15%), and patch usability and explanation (10%).

`{{item.reference_patch}}`, `{{item.test_expectations}}`, and `{{item.requirements}}` refer to dataset fields. `{{sample.output_text}}` is the model's complete response. The grader reviews the proposed repair; it does **not** apply the patch or execute tests.

In [ ]:
GRADER = json.loads(
    (DEMO_ROOT / "graders" / "code_repair_quality.json").read_text(encoding="utf-8")
)
GRADER

## 4. Connect to your Foundry project

Copy [`.env.template`](../.env.template) to a demo-local `.env` and set `FOUNDRY_PROJECT_ENDPOINT`. Authenticate with a provider supported by `DefaultAzureCredential`, such as Azure CLI.

`AIProjectClient.get_openai_client()` provides the project's Files and Fine-tuning APIs. We disable automatic retries so a lost upload or submission response is not silently retried. Check Foundry before repeating either operation after an error.

This sample uses **Global training**. MAI supports Global and Developer, not Standard or Data zone; these training types do not guarantee residency in the project's region. Review [model and training-type support](https://learn.microsoft.com/azure/foundry/fine-tuning/overview#supported-models), [grader hosting requirements](https://learn.microsoft.com/azure/foundry/openai/how-to/reinforcement-fine-tuning#model-grader-deployments), and data terms before uploading.

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(DEMO_ROOT / ".env", override=False)
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=credential,
)
client = project_client.get_openai_client(max_retries=0)
OUTPUT_DIR.mkdir(exist_ok=True)

## 5. Upload the training and validation files

Upload both JSONL files with `purpose="fine-tune"`, then wait for processing. The files include the grader's reference fields; do not strip them before upload.

Each execution uploads new files. Their IDs are saved immediately in ignored `outputs/` files. If interrupted, retrieve those files with `client.files.retrieve(...)` instead of uploading again. If file quota is full, reconcile existing files rather than deleting shared data.

In [ ]:
(OUTPUT_DIR / "project-endpoint.txt").write_text(
    os.environ["FOUNDRY_PROJECT_ENDPOINT"], encoding="utf-8"
)
with TRAIN_PATH.open("rb") as data:
    training_file = client.files.create(file=data, purpose="fine-tune")
(OUTPUT_DIR / "training-file-id.txt").write_text(training_file.id, encoding="utf-8")

with VALIDATION_PATH.open("rb") as data:
    validation_file = client.files.create(file=data, purpose="fine-tune")
(OUTPUT_DIR / "validation-file-id.txt").write_text(validation_file.id, encoding="utf-8")

for uploaded_file in (training_file, validation_file):
    client.files.wait_for_processing(uploaded_file.id, max_wait_seconds=600)

print("Training and validation files are processed.")

## 6. Create the RFT job

Pass the uploaded file IDs and grader directly to `fine_tuning.jobs.create`. `method.type="reinforcement"` selects RFT. `trainingType="globalStandard"` selects Global training; omitting it defaults to Standard, which this model does not support.

Optional hyperparameters use model-specific service defaults. The grader's sampling seed is not a training seed. Review training and grading [pricing](https://aka.ms/oai/pricing) before submission.

**Run this cell once to create a new paid job.** To continue an existing job, skip this cell and use its saved ID in Section 7. If submission raises an error, check Foundry for the job before trying again.

In [ ]:
training_job = client.fine_tuning.jobs.create(
    model=MODEL,
    training_file=training_file.id,
    validation_file=validation_file.id,
    method={"type": "reinforcement", "reinforcement": {"grader": GRADER}},
    extra_body={"trainingType": "globalStandard"},
    suffix="mai-code-repair",
)
(OUTPUT_DIR / "job-id.txt").write_text(training_job.id, encoding="utf-8")
print(f"Job status: {training_job.status}")

## 7. Monitor training

Retrieve the job every minute until it succeeds, fails, or is cancelled. The six-hour timeout stops local waiting, **not** the remote job or its costs.

After a kernel restart, run Sections 1 and 4 with the **same project**, then continue here using `outputs/job-id.txt`. Do not rerun upload or submission to restart monitoring. For a job recovered through Foundry, save its confirmed ID to that file first.

In [ ]:
job_id = (OUTPUT_DIR / "job-id.txt").read_text(encoding="utf-8").strip()
deadline = time.monotonic() + 6 * 60 * 60

while True:
    job = client.fine_tuning.jobs.retrieve(job_id)
    print(f"Job status: {job.status}")
    if job.status in {"succeeded", "failed", "cancelled"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("Monitoring timed out. The remote job may still be running.")
    time.sleep(60)

if job.status != "succeeded":
    raise RuntimeError(f"Training {job.status}: {job.error}")

(OUTPUT_DIR / "completed-job.json").write_text(job.model_dump_json(indent=2), encoding="utf-8")
print("Training completed. The model and result-file IDs are saved in outputs/completed-job.json.")

## 8. Inspect training metrics

`job.result_files` contains the service-produced training metrics. Download the CSVs with the Files API and view them as tables. Reward columns describe how the model scored against the grader during training; they are not executable-test pass rates.

In [ ]:
import pandas as pd
from IPython.display import display

if not job.result_files:
    raise RuntimeError("The service returned no training metrics files.")

for index, file_id in enumerate(job.result_files, start=1):
    result = client.files.content(file_id)
    metrics_path = OUTPUT_DIR / f"training-metrics-{index}.csv"
    result.write_to_file(metrics_path)
    display(pd.read_csv(metrics_path))

## 9. Close local clients and plan evaluation

The trained artifact is `job.fine_tuned_model`; it is **not** a serving deployment. Keep the saved IDs, completed-job response, and metric CSVs in ignored `outputs/`, not in committed notebook output.

The 15 validation tasks are used during training, not held out for a final quality claim. Before relying on the model, calibrate the grader and compare repairs on an independent test set using isolated executable tests and human review.

Closing these clients releases local connections only. It does not cancel training or delete uploaded files, model artifacts, or any separately provisioned grader deployment. Follow [post-training guidance](../../../POST_TRAINING.md) for evaluation, serving, cancellation, and cleanup.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.

In [ ]:
client.close()
project_client.close()
credential.close()